In [1]:
# 📦 Install required libraries
%pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
%pip install langchain-groq
%pip install sentence-transformers
%pip install langchain_openai
%pip install "setuptools<=80.10.2"

  Using cached regex-2023.12.25-cp310-cp310-win_amd64.whl.metadata (41 kB)
Using cached regex-2023.12.25-cp310-cp310-win_amd64.whl (269 kB)
  Attempting uninstall: regex
    Found existing installation: regex 2026.7.10
    Uninstalling regex-2026.7.10:
      Successfully uninstalled regex-2026.7.10
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.13.0 requires regex>=2025.10.22, but you have regex 2023.12.25 which is incompatible.


Note: you may need to restart the kernel to use updated packages.
  Using cached regex-2026.7.10-cp310-cp310-win_amd64.whl.metadata (41 kB)
Using cached regex-2026.7.10-cp310-cp310-win_amd64.whl (277 kB)
  Attempting uninstall: regex
    Found existing installation: regex 2023.12.25
    Uninstalling regex-2023.12.25:
      Successfully uninstalled regex-2023.12.25
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 0.28.8 requires regex<2024.0.0,>=2023.12.25, but you have regex 2026.7.10 which is incompatible.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 📥 Import necessary packages and modules
from langchain_openai import ChatOpenAI
import os as os
from crewai_tools import PDFSearchTool
from langchain_community.tools.tavily_search import TavilySearchResults
from crewai_tools  import tool
from crewai import Crew
from crewai import Task
from crewai import Agent
import matplotlib.pyplot as plt
import numpy as np
import pkg_resources

C:\Users\Rayha\Downloads\lib\site-packages\crewai\telemetry\telemetry.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
C:\Users\Rayha\Downloads\lib\site-packages\pydantic\_internal\_generate_schema.py:949: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warnings.warn(


In [3]:
# 🔐 Load API keys
GROQ_API_KEY='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'
TAVILY_API_KEY='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'

import os
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

In [4]:
# 🧠 Set up the LLM model using Groq's Llama3
llm = ChatOpenAI(
    openai_api_base="https://api.groq.com/openai/v1",
    openai_api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant", #updated model as Groq decommissioned old version
    temperature=0.1,
    max_tokens=1000,
)

In [5]:
# 🌐 Set up Tavily tool for real-time web search
web_search_tool = TavilySearchResults(k=3)

In [6]:
# 🔎 Run a web search using the Tavily tool
web_search_tool.run("What does Sporo Health do?")

[{'url': 'https://envolveglobal.org/envolvexl-portfolio/sporo-health',
  'content': '## Sporo Health\n\nSporo Health is an AI research lab for healthcare. Workforce shortages, high AI implementation costs, and slow adoption — along with data security concerns and AI inaccuracies—hinder the scalability and effectiveness of healthcare delivery. Addressing these challenges is essential to enhance efficiency and improve patient outcomes. Searching through hundreds of patient chart documents and puzzle-piecing together a history is time-consuming and difficult – let Sporo do it for you within seconds, focusing on the key elements of a patient’s story in an easily-digestible, familiar, presentation format.\n\n#### Follow Us\n\n#### Menu\n\n#### Newsletter\n\n©2026 Envolve Entrepreneurship. All rights reserved. Developed by Cactus'},
 {'url': 'https://info.sporo.health',
  'content': "Dr. Wael Aboughali MD, Family Medicine\n\nI have been a family physician for two decades and Sporo Health is 

In [7]:
@tool
def router_tool(question, **kwargs):
  """Router Function"""
  if 'Sporo Health' in question:
    return 'vectorstore'
  else:
    return 'web_search'

In [8]:
# 🤖 Defining AI agents with their roles and toolsets
#Router, Retriever, Grader, Hallucination, Answer Grader 

Router_Agent = Agent(
  role='Router',
  goal='Route user question to a vectorstore or web search',
  backstory=(
    "You are an expert at routing a user question to a vectorstore or web search."
    "Use the vectorstore for questions on concept related to Retrieval-Augmented Generation."
    "You do not need to be stringent with the keywords in the question related to these topics. Otherwise, use web-search."
  ),
  verbose=True,
  allow_delegation=False,
  llm=llm,
)

In [9]:
Retriever_Agent = Agent(
role="Retriever",
goal="Use the information retrieved from the vectorstore to answer the question",
backstory=(
    "You are an assistant for question-answering tasks."
    "Use the information present in the retrieved context to answer the question."
    "You have to provide a clear concise answer."
),
verbose=True,
allow_delegation=False,
llm=llm,
)

In [10]:
Grader_agent =  Agent(
  role='Answer Grader',
  goal='Filter out erroneous retrievals',
  backstory=(
    "You are a grader assessing relevance of a retrieved document to a user question."
    "If the document contains keywords related to the user question, grade it as relevant."
    "It does not need to be a stringent test.You have to make sure that the answer is relevant to the question."
  ),
  verbose=True,
  allow_delegation=False,
  llm=llm,
)

In [11]:
hallucination_grader = Agent(
    role="Hallucination Grader",
    goal="Filter out hallucination",
    backstory=(
        "You are a hallucination grader assessing whether an answer is grounded in / supported by a set of facts."
        "Make sure you meticulously review the answer and check if the response provided is in alignmnet with the question asked"
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [12]:
answer_grader = Agent(
    role="Answer Grader",
    goal="Filter out hallucination from the answer.",
    backstory=(
        "You are a grader assessing whether an answer is useful to resolve a question."
        "Make sure you meticulously review the answer and check if it makes sense for the question asked"
        "If the answer is relevant generate a clear and concise response."
        "If the answer generated is not relevant then perform a websearch using 'web_search_tool'"
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [13]:
## DEFINE NEW AGENT ROLE
referencing_agent = Agent(
    role='Referencing',
    goal="Cite original sources of the answer.",
    backstory=(
        "You are an assistant for citing original source material of information provided in the answer."
        "Make sure you provide the URL to any websites where the relevant answer information was extracted from."
        "The source material must be displayed in a Harvard style referencing format underneath each answer."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [14]:
## DEFINE GENERATOR AGENT
generator_agent = Agent(
    role="Senior report writer and analyst",
    goal="Compile search answers and references into clean, professional reports.",
    backstory=(
        "You are an elite technical writer. You take raw information and "
        "academic citations, extract the most important insights, and organize "
        "them into easy-to-read Markdown reports."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)       

In [15]:
tavily_tool = TavilySearchResults(max_results=3)

In [16]:
# 🧩 Defining the task each agent is responsible for
router_task = Task(
    description=("Analyse the keywords in the question {question}"
    "Based on the keywords decide whether it is eligible for a vectorstore search or a web search."
    "Return a single word 'vectorstore' if it is eligible for vectorstore search."
    "Return a single word 'websearch' if it is eligible for web search."
    "Do not provide any other premable or explanation."
    ),
    expected_output=("Give a binary choice 'websearch' or 'vectorstore' based on the question"
    "Do not provide any other premable or explaination."),
    agent=Router_Agent,
    tools=[router_tool],
)

In [17]:
retriever_task = Task(
    description=("Based on the response from the router task extract information for the question {question} with the help of the respective tool."
    "Use the web_serach_tool to retrieve information from the web in case the router task output is 'websearch'."
    "Use the rag_tool to retrieve information from the vectorstore in case the router task output is 'vectorstore'."
    ),
    expected_output=("You should analyse the output of the 'router_task'"
    "If the response is 'websearch' then use the web_search_tool to retrieve information from the web."
    "If the response is 'vectorstore' then use the rag_tool to retrieve information from the vectorstore."
    "Return a claer and consise text as response."),
    agent=Retriever_Agent,
    context=[router_task],
    #tools=[retriever_tool],
)

In [18]:
grader_task = Task(
    description=("Based on the response from the retriever task for the quetion {question} evaluate whether the retrieved content is relevant to the question."
    ),
    expected_output=("Binary score 'yes' or 'no' score to indicate whether the document is relevant to the question"
    "You must answer 'yes' if the response from the 'retriever_task' is in alignment with the question asked."
    "You must answer 'no' if the response from the 'retriever_task' is not in alignment with the question asked."
    "Do not provide any preamble or explanations except for 'yes' or 'no'."),
    agent=Grader_agent,
    context=[retriever_task],
)

In [19]:
hallucination_task = Task(
    description=("Based on the response from the grader task for the question {question} evaluate whether the answer is grounded in / supported by a set of facts."),
    expected_output=("Binary score 'yes' or 'no' score to indicate whether the answer is sync with the question asked"
    "Respond 'yes' if the answer is in useful and contains fact about the question asked."
    "Respond 'no' if the answer is not useful and does not contains fact about the question asked."
    "Do not provide any preamble or explanations except for 'yes' or 'no'."),
    agent=hallucination_grader,
    context=[grader_task],
)

In [20]:
answer_task = Task(
    description=("Based on the response from the hallucination task for the question {question} evaluate whether the answer is useful to resolve the question."
    "If the answer is 'yes' return a clear and concise answer."
    "If the answer is 'no' then perform a 'websearch' and return the response"),
    expected_output=("Return a clear and concise response if the response from 'hallucination_task' is 'yes'."
    "Perform a web search using 'web_search_tool' and return ta clear and concise response only if the response from 'hallucination_task' is 'no'."
    "Otherwise respond as 'Sorry! unable to find a valid response'."),
    context=[hallucination_task],
    agent=answer_grader,
    #tools=[answer_grader_tool],
)

In [21]:
## NEW AGENT TASKS HERE

referencing_task = Task(
    description=("Based on the response from the answer task for the question {question}, cite the original sources of the information used in formulating the answer."),
    expected_output=("If the source material has a valid website URL, publishing author names, a title, a publishing date and access date, generate a Harvard style citation."
                      "If any of these features of the source material are unknown, generate the reference as normal without these specific features."
                     "Generate seperate citations for each relevant piece of information in the answer."),
    context=[answer_task],
    agent=referencing_agent,
    tools=[tavily_tool],
)                

In [22]:
generator_task = Task(
    description=("Based on the response from the answer and referencing tasks for the question {question}, generate a report summary from key information."),
    expected_output=("If a clear answer with source citations are provided, summarise key insights from the information and export a structured report from it."
                     "The exported report must be structured in a PDF or Markdown file format."
                     "If no clear answer and no source citations are provided, do not summarise or export any information."),
    context=[answer_task, referencing_task],
    agent=generator_agent,
    tools=[],
    output_file="structured_report.md",
)      

In [23]:
# 👥 Assemble agents into a Crew to collaborate on the task
rag_crew = Crew(
    agents=[Router_Agent, Retriever_Agent, Grader_agent, hallucination_grader, answer_grader, referencing_agent, generator_agent],
    tasks=[router_task, retriever_task, grader_task, hallucination_task, answer_task, referencing_task, generator_task],
    verbose=True,

)

In [24]:
inputs ={"question":"Does Sporo Streamline patient chart reviews?"}

In [ ]:
# ▶️ Start the collaborative process among the agents
result = rag_crew.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Router
 [INFO]: == Starting Task: Analyse the keywords in the question Does Sporo Streamline patient chart reviews?Based on the keywords decide whether it is eligible for a vectorstore search or a web search.Return a single word 'vectorstore' if it is eligible for vectorstore search.Return a single word 'websearch' if it is eligible for web search.Do not provide any other premable or explanation.


> Entering new CrewAgentExecutor chain...
Thought: I need to determine whether the question is related to Retrieval-Augmented Generation to decide whether to use the vectorstore or web search.

Action: router_tool
Action Input: {"question": "Does Sporo Streamline patient chart reviews?"} 

web_search

Thought: The question is not related to Retrieval-Augmented Generation, so I should use web search.

Action: router_tool
Action Input: {"question": "Does Sporo Streamline patient chart reviews?"} 

I tried reusing the same input, I must stop using this action input. 

In [ ]:
print(result)